In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


class FinancialRatiosTransformer(BaseEstimator, TransformerMixin):
    """
    Computes key domain-specific financial ratios for Home Credit data.
    Adds feature engineering directly inside the scikit-learn pipeline.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()


        income = X_out['AMT_INCOME_TOTAL'].replace(0, np.nan)
        credit = X_out['AMT_CREDIT'].replace(0, np.nan)

        X_out['CREDIT_INCOME_PERCENT'] = X_out['AMT_CREDIT'] / income
        X_out['ANNUITY_INCOME_PERCENT'] = X_out['AMT_ANNUITY'] / income
        X_out['CREDIT_TERM'] = X_out['AMT_ANNUITY'] / credit
        X_out['DAYS_EMPLOYED_PERCENT'] = X_out['DAYS_EMPLOYED'] / X_out['DAYS_BIRTH'].replace(0, np.nan)

        return X_out



if __name__ == "__main__":
    # --- Load Data ---
    df = pd.read_csv("data/application_train.csv")

    # Drop identifier and target column
    X = df.drop(columns=["SK_ID_CURR", "TARGET"])
    y = df["TARGET"]

    # Identify categorical and numerical features dynamically
    categorical_cols = X.select_dtypes(include=["object", "category", "string", "str"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

    # Account for columns created by FinancialRatiosTransformer
    new_ratio_cols = [
        "CREDIT_INCOME_PERCENT",
        "ANNUITY_INCOME_PERCENT",
        "CREDIT_TERM",
        "DAYS_EMPLOYED_PERCENT"
    ]
    numerical_cols.extend(new_ratio_cols)


    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, numerical_cols),
        ("cat", cat_pipeline, categorical_cols)
    ])

    #Model 1: Conservative LightGBM
    lgbm_pipeline = Pipeline([
        ("ratios", FinancialRatiosTransformer()),
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(
            n_estimators=80,
            learning_rate=0.03,
            num_leaves=12,
            max_depth=4,
            min_child_samples=25,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=1.0,  # Keeping 1.0 preserves accurate rank-ordering for ROC-AUC
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])

    #Model 2: Logistic Regression
    lr_pipeline = Pipeline([
        ("ratios", FinancialRatiosTransformer()),
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            C=0.05,
            max_iter=1000,
            random_state=42
        ))
    ])

    # --- Cross-Validation Setup ---
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("Evaluating LightGBM Classifier...")
    lgbm_oof_preds = cross_val_predict(lgbm_pipeline, X, y, cv=skf, method="predict_proba")[:, 1]
    lgbm_auc = roc_auc_score(y, lgbm_oof_preds)
    lgbm_gini = 2 * lgbm_auc - 1

    print("\nEvaluating Logistic Regression Baseline...")
    lr_oof_preds = cross_val_predict(lr_pipeline, X, y, cv=skf, method="predict_proba")[:, 1]
    lr_auc = roc_auc_score(y, lr_oof_preds)
    lr_gini = 2 * lr_auc - 1

    print("\n" + "="*40)
    print("Results metric summary")
    print(f"LightGBM -> ROC-AUC: {lgbm_auc:.4f} | Gini: {lgbm_gini:.4f}")
    print(f"LogReg   -> ROC-AUC: {lr_auc:.4f} | Gini: {lr_gini:.4f}")
    print("="*40)

Evaluating LightGBM Classifier...

Evaluating Logistic Regression Baseline...

Results metric summary
LightGBM -> ROC-AUC: 0.4589 | Gini: -0.0823
LogReg   -> ROC-AUC: 0.5343 | Gini: 0.0685
